# Figure 15 — the same twenty experiments, spent three ways

Grid, a face-centred central composite design with a fitted quadratic response surface, and Bayesian optimization. Identical budget, identical landscape.

**The honest finding:** in two dimensions with twenty runs, all three get close. A grid is hard to beat on a small, cheap, low-dimensional problem — which is exactly what the lecture says. The difference BO makes is HOW FAST it gets there, shown in panel d, and that gap widens with every factor you add. For real benchmarks see Felton et al., Chemistry-Methods 2021, 1, 116-122.


In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent / "_shared"))
sys.path.insert(0, str(pathlib.Path.cwd() / "_shared"))
import numpy as np, matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import style, gp as gpmod, landscape as land, doe as doemod
style.use_deck_style()
OUT = "../lecture_12_figures/generated"

In [ ]:

style.SHOW_TEXT = True   # False -> clean panels (no titles/captions/labels)
                         # for ungrouping the PDF into editable pptx shapes

T, C, Z = land.mesh()
xstar, zstar = land.optimum()
BUDGET = 20
lo, hi = land.BOUNDS[:, 0], land.BOUNDS[:, 1]
print("true optimum", np.round(xstar, 1), "=", round(zstar, 1), "%")

fig, axes = plt.subplots(1, 3, figsize=(style.FIG_W_FULL, 3.6), sharey=True,
                         gridspec_kw=dict(wspace=0.15))


def base(ax):
    cs = ax.contourf(T, C, Z, levels=18, cmap="BuGn", alpha=0.9)
    ax.contour(T, C, Z, levels=8, colors="white", linewidths=0.4, alpha=0.6)
    ax.plot(*xstar, "*", ms=15, color=style.RED, mec="white", mew=0.8, zorder=8)
    style.xlabel(ax, land.LABELS[0])
    ax.set_xticks([70, 100, 130])
    return cs


def running_best(pts):
    return np.maximum.accumulate(land.yield_surface(np.asarray(pts)))


# ---- a. grid ---------------------------------------------------------------
gt = np.linspace(lo[0] + 8, hi[0] - 8, 5)
gc = np.linspace(lo[1] + 0.4, hi[1] - 0.4, 4)
grid = np.array([[a, b] for a in gt for b in gc])
cs = base(axes[0])
axes[0].scatter(grid[:, 0], grid[:, 1], s=28, c=style.INK, edgecolor="white",
                linewidth=0.7, zorder=6)
b0 = land.yield_surface(grid).max()
style.title(axes[0], f"a · grid — {b0:.0f}%", loc="left", color=style.INK,
            fontsize=10.5)
style.ylabel(axes[0], land.LABELS[1])

# ---- b. face-centred CCD + quadratic RSM (see _shared/doe.py) --------------
res_doe = doemod.run(land.BOUNDS, land.yield_surface, BUDGET, seed=0)
doe_pts, yv, ZH = res_doe["X"], res_doe["y"], res_doe["predict"](T, C)
cs = base(axes[1])
axes[1].contour(T, C, ZH, levels=7, colors=style.RED, linewidths=0.8,
                linestyles="--", alpha=0.95)
axes[1].scatter(doe_pts[:, 0], doe_pts[:, 1], s=28, c=style.INK, edgecolor="white",
                linewidth=0.7, zorder=6)
i = np.unravel_index(np.argmax(ZH), ZH.shape)
axes[1].plot(T[i], C[i], "P", ms=10, color=style.RED, mec="white", mew=0.8,
             zorder=9)
b1 = yv.max()
style.title(axes[1], f"b · CCD + quadratic — {b1:.0f}%", loc="left",
            color=style.INK, fontsize=10.5)
style.text(axes[1], 0.03, 0.04, "dashed: the fitted quadratic\ncannot bend along the "
           "ridge,\nso its predicted optimum (+)\nis in the wrong place",
           transform=axes[1].transAxes, fontsize=8.4, color=style.INK,
           va="bottom", bbox=dict(fc="white", alpha=0.85, ec="none", pad=2.0))
err = zstar - land.f([T[i], C[i]])
print(f"RSM predicted optimum is worth {land.f([T[i], C[i]]):.1f}% "
      f"— {err:.1f} points below the true optimum")

# ---- c. BO -----------------------------------------------------------------
gx = np.linspace(*land.BOUNDS[0], 60)
gy = np.linspace(*land.BOUNDS[1], 60)
cand = np.array([[a, b] for a in gx for b in gy])
BO_SEEDS = 12
runs = [gpmod.bo_loop(land.f, land.BOUNDS, n_init=6, n_iter=BUDGET - 6,
                      acq="ei", noise=0.0, seed=s, grid=cand,
                      ls=[20.0, 1.2], sf=28.0) for s in range(BO_SEEDS)]
curves = np.array([np.maximum.accumulate(r["ytrue"]) for r in runs])
res = runs[0]                      # panel c shows seed 0, stated in the caption
P = res["X"]
cs = base(axes[2])
axes[2].plot(P[6:, 0], P[6:, 1], "-", color=style.RED, lw=0.7, alpha=0.28)
axes[2].scatter(P[:6, 0], P[:6, 1], s=30, marker="s", c=style.INK,
                edgecolor="white", linewidth=0.7, zorder=6, label="6 seeds")
axes[2].scatter(P[6:, 0], P[6:, 1], s=28, c=style.RED, edgecolor="white",
                linewidth=0.7, zorder=7, label="14 by EI")
b2 = res["ytrue"].max()
style.title(axes[2], f"c · Bayesian optimization — {b2:.0f}%", loc="left",
            color=style.RED, fontsize=10.5)
style.text(axes[2], 0.03, 0.04, "one run (seed 0);\npanel d shows all 12",
           transform=axes[2].transAxes, fontsize=8.4, color=style.INK,
           va="bottom", bbox=dict(fc="white", alpha=0.85, ec="none", pad=2.0))
style.legend(axes[2], loc="upper left", fontsize=8.2, framealpha=0.9,
             facecolor="white")

cb = fig.colorbar(cs, ax=axes, fraction=0.016, pad=0.012)
style.cbar_label(cb, "true yield / %")
cb.outline.set_visible(False)
style.save(fig, "fig_15_grid_doe_bo", OUT)
plt.close(fig)

# ---- separate figure: sequential vs batch ----------------------------------
fig2, ax = plt.subplots(figsize=(6.2, 3.6))
bo_curve = np.median(curves, axis=0)
q1, q3 = np.percentile(curves, [25, 75], axis=0)
nn = np.arange(1, BUDGET + 1)
ax.fill_between(nn, q1, q3, color=style.RED, alpha=0.15, lw=0)
ax.plot(nn, bo_curve, color=style.RED, lw=2.3,
        label=f"BO — sequential (median of {BO_SEEDS} runs)")
ax.axhline(b0, color=style.INK, lw=1.4, ls="--",
           label=f"grid — batch, result at run 20 ({b0:.0f}%)")
ax.plot([BUDGET], [b0], "o", ms=8, color=style.INK)
ax.axhline(b1, color=style.GOLD, lw=1.4, ls="--",
           label=f"CCD + quadratic — batch ({b1:.0f}%)")
ax.plot([BUDGET], [b1], "o", ms=8, color=style.GOLD)
ax.axhline(zstar, color=style.INK, ls=":", lw=1.0)
style.text(ax, BUDGET, zstar + 1.2, "true optimum", ha="right", fontsize=8.6,
           color=style.INK)
cross = np.where(bo_curve >= b0)[0]
if len(cross):
    n_x = int(cross[0]) + 1
    ax.plot([n_x], [bo_curve[n_x - 1]], "o", ms=10, mfc="none", mec=style.RED,
            mew=2.0, zorder=8)
    style.annotate(ax, f"BO reaches the grid's answer\nat experiment {n_x}",
                   (n_x, bo_curve[n_x - 1]), textcoords="offset points",
                   xytext=(-10, -46), fontsize=8.8, color=style.RED, ha="right",
                   arrowprops=dict(arrowstyle="->", color=style.RED, lw=1.0))
    print(f"BO reaches the grid's best after {n_x} of 20 experiments (median)")
style.xlabel(ax, "experiment number")
style.ylabel(ax, "best true yield so far / %")
ax.set_ylim(0, 102)
ax.set_xlim(1, BUDGET + 0.5)
style.legend(ax, loc="lower right", fontsize=8.4)
style.title(ax, "a batch design tells you at the end; BO tells you as it goes",
            loc="left", fontsize=10.5)
style.save(fig2, "fig_15d_sequential_vs_batch", OUT)